In [0]:
from datetime import date

In [0]:
dbutils.widgets.text("load_date", "")
load_date = dbutils.widgets.get("load_date")


if load_date:
    load_date=date.fromisoformat(load_date)
else:
    load_date=date.today()

In [0]:
# Stores patient observation details.
patient_observation=spark.sql(f'''select p.id as patient_id,p.gender,p.birthDate,o.id as observation_id,o.status as observation_status,o.code.text as observation_name,o.code.coding[0].code as observation_code,o.code.coding[0].display as observation_display,coalesce(cast(o.valueBoolean as string),o.valueCodeableConcept.text) as observation_value,o.effectiveDateTime as observation_date,o.category,o.interpretation,o.referenceRange,o.meta.lastUpdated as observation_last_updated,cast('{load_date}' as timestamp) as gold_processed_timestamp
from workspace.silver.silver_patient p inner join workspace.silver.silver_observation o
on o.subject.reference=concat('Patient/',p.id) where p.is_current=true and o.is_current=true and p.id is not null and o.id is not null''')

patient_observation.write.mode('overwrite').saveAsTable('workspace.gold.gold_patient_observation')

In [0]:
# Stores patient condition details.
patient_condition = spark.sql(f'''select p.id as patient_id,p.gender,p.birthDate,c.id as condition_id,c.clinicalStatus.coding[0].code as clinical_status_code,c.verificationStatus.coding[0].code as verification_status_code,c.code.text as condition_name,c.code.coding[0].code as condition_code,c.onsetDateTime as condition_onset_date,c.severity,c.bodySite,c.meta.lastUpdated as condition_last_updated,cast('{load_date}' as timestamp) as gold_processed_timestamp
from workspace.silver.silver_patient p inner join workspace.silver.silver_condition c on c.subject.reference=concat('Patient/',p.id) where p.is_current=true and c.is_current=true and p.id is not null and c.id is not null''')

patient_condition.write.mode('overwrite').saveAsTable('workspace.gold.gold_patient_condition')

In [0]:
# Stores patient encounter details.
patient_encounters = spark.sql(f'''select p.id as patient_id,p.gender,p.birthDate,e.id as encounter_id,e.status as encounter_status,e.class.code as encounter_class_code,e.class.display as encounter_class_display,e.type,e.period.start as encounter_start,e.period.end as encounter_end,e.participant,e.serviceProvider,e.reasonCode,e.diagnosis,e.location,e.meta.lastUpdated as encounter_last_updated,cast('{load_date}' as timestamp) as gold_processed_timestamp from workspace.silver.silver_patient p inner join workspace.silver.silver_encounter e on e.subject.reference=concat('Patient/',p.id) where p.is_current=true and e.is_current=true and p.id is not null and e.id is not null''')

patient_encounters.write.mode('overwrite').saveAsTable('workspace.gold.gold_patient_encounter')

In [0]:
# Stores patient demographics and clinical record counts.
patient_summary = spark.sql(f'''with encounter_counts as (select subject.reference as patient_reference,count(distinct id) as encounter_count from workspace.silver.silver_encounter where is_current = TRUE and subject.reference is not null group by subject.reference),

observation_counts as (select subject.reference as patient_reference,count(distinct id) as observation_count from workspace.silver.silver_observation where is_current = TRUE and subject.reference is not null group by subject.reference),

condition_counts as (select subject.reference as patient_reference,count(distinct id) as condition_count from workspace.silver.silver_condition where is_current = TRUE and subject.reference is not null group by subject.reference)

select p.id as patient_id,p.gender,p.birthDate,p.maritalStatus.text as marital_status,coalesce(e.encounter_count,0) as encounter_count,coalesce(o.observation_count,0) as observation_count,coalesce(c.condition_count,0) as condition_count,cast('{load_date}' as timestamp) as gold_processed_timestamp from workspace.silver.silver_patient p left join encounter_counts e on e.patient_reference=concat('Patient/',p.id) left join observation_counts o on o.patient_reference=concat('Patient/',p.id) left join condition_counts c on c.patient_reference=concat('Patient/',p.id) where p.is_current = TRUE''')

patient_summary.write.mode('overwrite').saveAsTable('workspace.gold.gold_patient_summary')

In [0]:
# Stores condition-level patient and occurrence counts.
condition_summary = spark.sql(f'''select c.code.text as condition_name,c.code.coding[0].code as condition_code,c.code.coding[0].display as condition_display,count(distinct c.id) as condition_count,count(distinct c.subject.reference) as patient_count,cast('{load_date}' as timestamp) as gold_processed_timestamp
from workspace.silver.silver_condition c where c.is_current=true and c.id is not null group by c.code.text,c.code.coding[0].code,c.code.coding[0].display''')

condition_summary.write.mode('overwrite').saveAsTable('workspace.gold.gold_condition_summary')

In [0]:
dbutils.notebook.exit('success')    